# Notebook 09 · Evaluation of the corrected RetinaNet (deviation 1, amendment v1.2)

**Why it exists:** the RetinaNet of the first run was trained with transposed boxes; notebook 08 retrained
the five models with the correction. Here the corrected RetinaNet (A′) is evaluated with the analyses of
notebook 07, against the predictions of the YOLO arms of the first run, which did not change. Nothing here
selects a model, tunes a parameter or changes the decision rule.

**What it does:**

1. Checks that the five jobs of notebook 08 finished with the corrected code (`a6f3c0fc5798879d`) and that
   the ten YOLO jobs (folder `completo/`) are those of the first run (`c3752d247a0173f1`).
2. Uses the five A′ models to predict the test set (CT-ICH) and the equivocal examinations of their fold.
3. Computes, in cross-validation, all the analyses of the plan with A′. The comparison between the two YOLO
   arms, which does not involve RetinaNet, must reproduce the first-run value.
4. Computes on the test set, **once**, the comparisons involving A′ (A′ versus B, A′ versus C and the drop
   from validation to test of A′). Those of the first run without RetinaNet (C versus B and the YOLO
   drops) are read from `completo/results_test.json`, not recomputed.
5. Measures the CPU prediction time of A′.

Output goes to `dados/tc/experimento/correcao_retinanet/`; the folder `completo/` is only read.

**Before running:** the five jobs finished (cell 4 of notebook 08 reporting 5 of 5); `src/ichdet`
**version 0.4.0** on the Drive as `MyDrive/ich-benchmark/codigo/ichdet/`; L4 GPU.

**Time:** about 40 minutes.

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Drive, libraries, GPU and check of the frozen code
import json, os, platform, socket, subprocess, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
    DISCO = Path('/content')                         # Colab disk: reading many images from the Drive is slow
    from importlib.metadata import version
    numpy_antes = version('numpy')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'monai==1.5.2', 'ultralytics>=8.3,<8.4'],
                   check=True)
    if version('numpy') != numpy_antes:
        raise SystemExit(f'O pip trocou o numpy ({numpy_antes} -> {version("numpy")}). '
                         'Reinicie a sessão (Ambiente de execução → Reiniciar sessão) e rode de novo.')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['EXPERIMENTO_LOCAL'])
    DISCO = RAIZ / 'disco_local'

CODIGO = RAIZ / 'codigo'
sys.path.insert(0, str(CODIGO))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
PRIMEIRA = RAIZ / 'dados' / 'tc' / 'experimento' / 'completo'          # first run: read only
SAIDA = RAIZ / 'dados' / 'tc' / 'experimento' / 'correcao_retinanet'   # corrected RetinaNet and results
CQ500, CTICH = DISCO / 'cq500_512', DISCO / 'ctich_512'

import numpy as np, torch, monai, ultralytics
import ichdet
from ichdet import data_io, evaluate, experiment, final, training_config as cfg

CONGELADO = 'a6f3c0fc5798879d'                             # code of protocol v1.2 (corrected RetinaNet)
CONGELADO_PRIMEIRA = 'c3752d247a0173f1'              # code of protocol v1.1 (first run)
AMBIENTE = {
    'ichdet_version': ichdet.__version__, 'frozen_code_fingerprint': experiment.frozen_fingerprint(CODIGO / 'ichdet'),
    'python': sys.version.split()[0], 'numpy': np.__version__, 'torch': torch.__version__, 'monai': monai.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'cpu': platform.processor() or platform.machine(), 'cpu_count': os.cpu_count(),
    'session': f'{socket.gethostname()}-{os.getpid()}', 'started': time.strftime('%Y-%m-%d %H:%M:%S'),
}
print(json.dumps(AMBIENTE, indent=2))
assert AMBIENTE['frozen_code_fingerprint'] == CONGELADO, (
    'O código no Drive não é o corrigido (0.4.0). Copie de novo a pasta src/ichdet do repositório.')
assert torch.cuda.is_available() or 'EXPERIMENTO_LOCAL' in os.environ, 'Ative a GPU (Ambiente de execução → Alterar o tipo).'

In [ ]:
# 2. The 15 jobs (corrected A′ + YOLO of the first run), the images and the reference boxes
A, B, C = 'retinanet', 'yolo_no_mosaic', 'yolo_mosaic'
K = experiment.FOLDS
PASTA = {A: SAIDA, B: PRIMEIRA, C: PRIMEIRA}
CODIGO_DO_BRACO = {A: CONGELADO, B: CONGELADO_PRIMEIRA, C: CONGELADO_PRIMEIRA}

trabalhos = {}
for k in K:
    for braco in (A, B, C):
        f = experiment.job_dir(PASTA[braco], k, braco) / 'job.json'
        assert f.exists(), f'rodada {k} / {braco}: treino não encontrado em {PASTA[braco].name}/'
        j = json.loads(f.read_text())
        assert j['epochs'] == cfg.EPOCHS, f'rodada {k} / {braco}: {j["epochs"]} épocas'
        assert j['environment']['frozen_code_fingerprint'] == CODIGO_DO_BRACO[braco], f'rodada {k} / {braco}: código'
        assert final.weights_path(PASTA[braco], k, braco).exists(), f'rodada {k} / {braco}: pesos não encontrados'
        trabalhos[(k, braco)] = j
print(f'{len(trabalhos)} treinos prontos: RetinaNet corrigido ({CONGELADO}) e YOLO da primeira rodada ({CONGELADO_PRIMEIRA})')

for pasta, arquivo in ((CQ500, 'cq500_512.zip'), (CTICH, 'ctich_512.zip')):
    if not (pasta / 'index.csv').exists():
        inicio = time.time()
        with zipfile.ZipFile(PREPARO / arquivo) as z:
            z.extractall(pasta)
        print(f'{arquivo}: descompactado em {time.time() - inicio:.0f} s')

import csv
parte = {}                                           # examination -> fold (five folds, by patient)
with open(CQ500 / 'index.csv', newline='') as f:
    for r in csv.DictReader(f):
        parte[r['exame']] = int(r['parte'])

REF = {k: data_io.load_monai_split(CQ500 / 'monai' / f'fold{k}.json') for k in K}
duvidosos = json.loads((CQ500 / 'monai' / 'equivocal.json').read_text())['validation']
DUVIDOSOS = {k: [e for e in duvidosos if parte[e['exam']] == k] for k in K}
REF_DUV = {k: {data_io.image_key(e['image']): (e['exam'], np.zeros((0, 4))) for e in DUVIDOSOS[k]} for k in K}
teste = json.loads((CTICH / 'monai' / 'external.json').read_text())['validation']
REF_TESTE = data_io.load_monai_split(CTICH / 'monai' / 'external.json')

esperado = json.loads((PREPARO / 'export_512_summary.json').read_text())
for k in K:
    assert len(REF[k]) == esperado['cq500_folds_main_analysis'][f'fold{k}']['slices'], k
assert sum(len(v) for v in DUVIDOSOS.values()) == esperado['cq500_equivocal_slices']
assert len(teste) == esperado['ct_ich']['slices']
print(f"validação: {sum(len(v) for v in REF.values())} imagens; duvidosos: {len(duvidosos)}; teste (CT-ICH): {len(teste)}")

In [ ]:
# 3. A′ predictions on the test set and the equivocal examinations (GPU; resumes after a disconnection). The YOLO ones already exist.
for k in K:
    d = experiment.job_dir(SAIDA, k, A)
    for nome, entradas, raiz in (('pred_test.csv', teste, CTICH), ('pred_equivocal.csv', DUVIDOSOS[k], CQ500)):
        if (d / nome).exists():
            continue
        inicio = time.time()
        p = final.predict(A, final.weights_path(SAIDA, k, A), entradas, raiz)
        data_io.save_predictions(d / f'{nome}.tmp', p)
        (d / f'{nome}.tmp').replace(d / nome)
        print(f'rodada {k} / {A}: {nome} em {time.time() - inicio:.0f} s')

def predicoes(k, braco, nome='pred_validation.csv'):
    return data_io.load_predictions(experiment.job_dir(PASTA[braco], k, braco) / nome)

PRED = {(k, b): predicoes(k, b) for k, b in trabalhos}
PRED_DUV = {(k, b): predicoes(k, b, 'pred_equivocal.csv') for k, b in trabalhos}
PRED_TESTE = {(k, b): predicoes(k, b, 'pred_test.csv') for k, b in trabalhos}
print('predições carregadas')

In [ ]:
# 4. Cross-validation with A′: primary comparison, effect of mosaic, sensitivity and per-subtype analyses
def rodadas(x, y, ref=REF, pred=PRED):
    return [(ref[k], pred[(k, x)], pred[(k, y)]) for k in K]

validacao = {'primary_A_vs_B': evaluate.compare_crossval(rodadas(A, B))}
validacao['mosaic_C_vs_B'] = evaluate.compare_crossval(rodadas(C, B))
validacao['mosaic_A_vs_C'] = evaluate.compare_crossval(rodadas(A, C))

# C versus B does not involve RetinaNet: it must equal the first run (same predictions, same bootstrap draws)
antes = json.loads((PRIMEIRA / 'results_validation.json').read_text())['mosaic_C_vs_B']
assert np.isclose(validacao['mosaic_C_vs_B']['equivalence_A_minus_B']['difference'],
                  antes['equivalence_A_minus_B']['difference']), 'predições dos YOLO diferentes das da primeira rodada'

# Sensitivity analysis 1: predictions of the five folds pooled in a single table
validacao['sensitivity_pooled'] = final.compare_pooled(rodadas(A, B))
# Sensitivity analysis 2: equivocal examinations added to validation as negatives
com_duv = {}
for braco in (A, B):
    for k in K:
        com_duv[(k, braco)] = final.merge((REF[k], PRED[(k, braco)]), (REF_DUV[k], PRED_DUV[(k, braco)]))
validacao['sensitivity_equivocal_as_negative'] = evaluate.compare_crossval(
    [(com_duv[(k, A)][0], com_duv[(k, A)][1], com_duv[(k, B)][1]) for k in K])
# Sensitivity analysis 3: without examinations with BHX boxes but negative for the majority of CQ500 readers
discordantes = ['CQ500-CT-0', 'CQ500-CT-134', 'CQ500-CT-176', 'CQ500-CT-187', 'CQ500-CT-224', 'CQ500-CT-234', 'CQ500-CT-24', 'CQ500-CT-257', 'CQ500-CT-283', 'CQ500-CT-284', 'CQ500-CT-288', 'CQ500-CT-291', 'CQ500-CT-31', 'CQ500-CT-337', 'CQ500-CT-340', 'CQ500-CT-345', 'CQ500-CT-351', 'CQ500-CT-374', 'CQ500-CT-380', 'CQ500-CT-391', 'CQ500-CT-398', 'CQ500-CT-414', 'CQ500-CT-483', 'CQ500-CT-486', 'CQ500-CT-56', 'CQ500-CT-63', 'CQ500-CT-67']                          # from cq500_series_escolhidas.csv (votos_ich <= 1, i.e. at most one positive read)
assert set(discordantes) <= set(parte)
sem = {k: final.subset(REF[k], PRED[(k, A)], discordantes) for k in K}
validacao['sensitivity_without_bhx_cq500_disagreement'] = evaluate.compare_crossval(
    [(sem[k][0], sem[k][1], final.subset(REF[k], PRED[(k, B)], discordantes)[1]) for k in K])
validacao['sensitivity_without_bhx_cq500_disagreement']['exams_removed'] = discordantes

# Sensitivity per haemorrhage subtype at 1 false positive per examination (exploratory)
tipos = {}
for k in K:
    tipos.update(final.box_subtypes(REF[k], CQ500 / 'boxes_512.csv'))
validacao['subtype_sensitivity_1fp'] = {b: final.subtype_sensitivity([(REF[k], PRED[(k, b)]) for k in K], tipos)
                                        for b in (A, B, C)}

(SAIDA / 'results_validation.json').write_text(json.dumps(validacao, indent=2, default=float))
p = validacao['primary_A_vs_B']['equivalence_A_minus_B']
print(f"AP@0.5 RetinaNet corrigido {validacao['primary_A_vs_B']['A']['ap50']:.3f}; YOLO sem mosaico {validacao['primary_A_vs_B']['B']['ap50']:.3f}")
print(f"Diferença {p['difference']:+.3f}; IC 90% [{p['ci90'][0]:+.3f}, {p['ci90'][1]:+.3f}]; margem ±{p['margin']}")
print('Leitura pré-especificada:', final.reading(p['ci90'], p['margin']))

In [ ]:
# 5. Test set (CT-ICH): comparisons with A′ computed ONCE; afterwards only read from the file
primeira_teste = json.loads((PRIMEIRA / 'results_test.json').read_text())
arquivo_teste = SAIDA / 'results_test.json'
if arquivo_teste.exists():
    resultado_teste = json.loads(arquivo_teste.read_text())
    print(f"resultado da base de teste lido do arquivo (calculado em {resultado_teste['computed']})")
else:
    modelos = {b: [PRED_TESTE[(k, b)] for k in K] for b in (A, B, C)}
    resultado_teste = {
        'computed': time.strftime('%Y-%m-%d %H:%M:%S'),
        'A_vs_B': evaluate.compare_test(REF_TESTE, modelos[A], modelos[B]),
        'A_vs_C': evaluate.compare_test(REF_TESTE, modelos[A], modelos[C]),
        'generalisation_drop': {A: final.generalisation_drop([(REF[k], PRED[(k, A)]) for k in K], REF_TESTE, modelos[A])},
    }
    arquivo_teste.write_text(json.dumps(resultado_teste, indent=2, default=float))
# From the first run, not recomputed: C versus B and the YOLO drops
resultado_teste_completo = dict(resultado_teste, C_vs_B=primeira_teste['C_vs_B'],
                                generalisation_drop={**resultado_teste['generalisation_drop'],
                                                     B: primeira_teste['generalisation_drop'][B],
                                                     C: primeira_teste['generalisation_drop'][C]},
                                first_run_computed=primeira_teste['computed'])
t = resultado_teste['A_vs_B']
print(f"Teste: AP@0.5 RetinaNet corrigido {t['A']['ap50']:.3f}; YOLO sem mosaico {t['B']['ap50']:.3f}; "
      f"diferença {t['equivalence_A_minus_B']['difference']:+.3f}")
for b, q in resultado_teste_completo['generalisation_drop'].items():
    print(f"{b:15s} validação {q['validation_ap50']:.3f} → teste {q['test_ap50']:.3f} (queda {q['drop']:+.3f})")

In [ ]:
# 6. CPU prediction time of A′ (fold-1 model, 100 CT-ICH images, one at a time)
latencia = {A: final.cpu_latency(A, final.weights_path(SAIDA, 1, A), teste, CTICH)}
q = latencia[A]
print(f"{A:15s} mediana {q['median_seconds_per_slice'] * 1000:.0f} ms por imagem ({q['torch_threads']} threads)")

In [ ]:
# 7. Single record of the final results
resumo = {'environment': AMBIENTE, 'deviation': 1, 'amendment': 'v1.2',
          'frozen_code': {'retinanet_corrected': CONGELADO, 'yolo_first_run': CONGELADO_PRIMEIRA,
                          'modules_changed': ['retinanet.py']},
          'jobs': len(trabalhos), 'validation': validacao, 'test': resultado_teste_completo,
          'cpu_latency': latencia}
(SAIDA / 'final_results.json').write_text(json.dumps(resumo, indent=2, default=float))
print('final_results.json gravado em', SAIDA)

## After running

1. `final_results.json`, `results_validation.json` and `results_test.json` in
   `dados/tc/experimento/correcao_retinanet/` hold the final results (included in this repository under
   `data/`).
2. **Do not run cell 5 again with different code:** the test-set result is the one computed the first time.